In [1]:
import pandas as pd
from pathlib import Path
from collections import Counter
import unicodedata
import os
import re
import zlib
import fitz
import olefile

## 데이터 불러오기

In [35]:
df = pd.read_csv("/home/bidcoin/data_cleaning2.csv", encoding="utf-8")
df.columns

Index(['공고 번호', '공고 차수', '사업명', '사업 금액', '발주 기관', '공개 일자', '입찰 참여 시작일',
       '입찰 참여 마감일', '사업 요약', '파일형식', '파일명', '텍스트', '텍스트길이'],
      dtype='str')

In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      100 non-null    str    
 1   공고 차수      82 non-null     float64
 2   사업명        100 non-null    str    
 3   사업 금액      100 non-null    float64
 4   발주 기관      100 non-null    str    
 5   공개 일자      100 non-null    str    
 6   입찰 참여 시작일  100 non-null    str    
 7   입찰 참여 마감일  93 non-null     str    
 8   사업 요약      100 non-null    str    
 9   파일형식       100 non-null    str    
 10  파일명        100 non-null    str    
 11  텍스트        100 non-null    str    
 12  텍스트길이      7 non-null      float64
dtypes: float64(3), str(10)
memory usage: 1.5 MB


### 텍스트 살펴보기 - hwp

In [37]:
df['발주 기관'].unique()

<ArrowStringArray>
[                      '한영대학',                     '한국연구재단',
                  '한국생산기술연구원',                      '인천광역시',
                   '경상북도 봉화군',                   '한국전기안전공사',
                  '재단법인충북연구원',                      '고려대학교',
                '재단법인스포츠윤리센터',                    '국방과학연구소',
              '(사）한국대학스포츠협의회',                   '한국사학진흥재단',
                    '서울시립대학교',                      '경희대학교',
                    '한국수자원공사',              '국가과학기술지식정보서비스',
                '한국철도공사 (용역)', '2025 구미 아시아육상경기선수권대회 조직위원회',
               '한국발명진흥회 입찰공고',                   '고양도시관리공사',
                      '전북대학교',                  '한국보건산업진흥원',
                  '한국사회보장정보원',                      '수협중앙회',
                    '한국농어촌공사',                 'KOICA 전자조달',
                   '대한장애인체육회',                   '인천광역시 동구',
                   '축산물품질평가원',                      '울산광역시',
                    '한국재정정보원',                     '부산관광공사',
     

In [38]:
df['발주 기관'].nunique()

87

In [39]:
len(df['발주 기관'])

100

In [40]:
# 기관유형별로 문서형식이 비슷하지 않을까 싶어 섹터 나눠보고자 함
# "중앙행정기관": central
# "지자체": local
# "공기업/공공기관": public
# "대학/교육기관": education
# "연구기관": research
# "재단/협회/비영리": nonprofit
# "민간기업": company

In [41]:
def get_org_sector_map():
    central = [
        "대검찰청",
        "중앙선거관리위원회",
    ]

    local = [
        "서울특별시교육청",
        "경기도 안양시",
        "경기도 평택시",
        "경상북도 봉화군",
        "인천광역시 동구",
        "서울특별시",
        "울산광역시",
        "인천광역시",
        "전북특별자치도 정읍시",
    ]

    public = [
        "국립인천해양박물관",
        "국가과학기술지식정보서비스",
        "BioIN",
        "재단법인경기도일자리재단",
        "서울특별시 여성가족재단",
        "재단법인 광주광역시 광주문화재단",
        "경기도사회서비스원",
        "세종테크노파크",
        "대한장애인체육회",
        "한국연구재단",
        "한국사학진흥재단",
        "한국보건산업진흥원",
        "한국사회보장정보원",
        "한국재정정보원",
        "한국로봇산업진흥원",
        "한국건강가정진흥원",
        "한국보육진흥원",
        "서민금융진흥원",
        "한국발명진흥회 입찰공고",
        "한국지식재산보호원",
        "축산물품질평가원",
        "한국교육과정평가원",
        "국립중앙의료원",
        "국가철도공단",
        "국민연금공단",
        "한국산업인력공단",
        "한국어촌어항공단",
        "한국산업단지공단",
        "한국전기안전공사",
        "한국수자원공사",
        "한국철도공사 (용역)",
        "고양도시관리공사",
        "부산관광공사",
        "파주도시관광공사",
        "한국농어촌공사",
        "한국농수산식품유통공사",
        "한국가스공사",
        "그랜드코리아레저(주)",
        "인천공항운영서비스(주)",
        "한국수출입은행",
        "KOICA 전자조달",
        "재단법인스포츠윤리센터",
        "한국해양조사협회",
        "대한적십자사 의료원",
        "(재)예술경영지원센터",
        "재단법인 한국장애인문화예술원",
        "문화체육관광부 국립민속박물관",
    ]

    education = [
        "경희대학교",
        "고려대학교",
        "광주과학기술원",
        "남서울대학교",
        "대전대학교",
        "서영대학교 산학협력단",
        "서울시립대학교",
        "을지대학교",
        "전북대학교",
        "조선대학교",
        "한영대학",
    ]

    research = [
        "국방과학연구소",
        "재단법인충북연구원",
        "재단법인 광주연구원",
        "한국생산기술연구원",
        "기초과학연구원",
        "한국한의학연구원",
        "한국원자력연구원",
        "나노종합기술원",
        "한국수자원조사기술원",
    ]

    nonprofit = [
        "대한상공회의소",
        "사단법인아시아물위원회사무국",
        "사단법인 보험개발원",
        "(사)벤처기업협회",
        "(사)부산국제영화제",
        "(사）한국대학스포츠협의회",
        "2025 구미 아시아육상경기선수권대회 조직위원회",
        "수협중앙회",
    ]

    company = [
        "케빈랩 주식회사",
    ]

    sector_groups = {
        "중앙행정기관(central)": central,
        "지자체(local)": local,
        "공기업/공공기관(public)": public,
        "대학/교육기관(education)": education,
        "연구기관(research)": research,
        "재단/협회/비영리(nonprofit)": nonprofit,
        "민간기업(company)": company,
    }

    org_sector_map = {
        org: sector
        for sector, org_list in sector_groups.items()
        for org in org_list
    }

    return org_sector_map

In [42]:
# 기관 매핑
df["기관 섹터"] = df["발주 기관"].map(get_org_sector_map())

In [43]:
print("기관섹터 null 개수:", df["기관 섹터"].isna().sum())

기관섹터 null 개수: 0


In [44]:
df["기관 섹터"].value_counts().sum()

np.int64(100)

In [45]:
# 매핑 검증
is_hwp = df["파일명"].str.lower().str.endswith(".hwp")

df.loc[is_hwp, ["기관 섹터", "발주 기관"]] \
  .drop_duplicates() \
  .sort_values(["기관 섹터", "발주 기관"]) \
  .groupby("기관 섹터")["발주 기관"] \
  .apply(list)

sector_orgs = (
    df[["기관 섹터", "발주 기관"]]
      .dropna(subset=["기관 섹터", "발주 기관"])
      .drop_duplicates()
      .sort_values(["기관 섹터", "발주 기관"])
      .groupby("기관 섹터")["발주 기관"]
      .apply(list)
)

for sector, orgs in sector_orgs.items():
    print(f"■ {sector} (총 {len(orgs)}개)")
    for org in orgs:
        print(f"  - {org}")
    print("-" * 50)


■ 공기업/공공기관(public) (총 47개)
  - (재)예술경영지원센터
  - BioIN
  - KOICA 전자조달
  - 경기도사회서비스원
  - 고양도시관리공사
  - 국가과학기술지식정보서비스
  - 국가철도공단
  - 국립인천해양박물관
  - 국립중앙의료원
  - 국민연금공단
  - 그랜드코리아레저(주)
  - 대한장애인체육회
  - 대한적십자사 의료원
  - 문화체육관광부 국립민속박물관
  - 부산관광공사
  - 서민금융진흥원
  - 서울특별시 여성가족재단
  - 세종테크노파크
  - 인천공항운영서비스(주)
  - 재단법인 광주광역시 광주문화재단
  - 재단법인 한국장애인문화예술원
  - 재단법인경기도일자리재단
  - 재단법인스포츠윤리센터
  - 축산물품질평가원
  - 파주도시관광공사
  - 한국가스공사
  - 한국건강가정진흥원
  - 한국교육과정평가원
  - 한국농수산식품유통공사
  - 한국농어촌공사
  - 한국로봇산업진흥원
  - 한국발명진흥회 입찰공고
  - 한국보건산업진흥원
  - 한국보육진흥원
  - 한국사학진흥재단
  - 한국사회보장정보원
  - 한국산업단지공단
  - 한국산업인력공단
  - 한국수자원공사
  - 한국수출입은행
  - 한국어촌어항공단
  - 한국연구재단
  - 한국재정정보원
  - 한국전기안전공사
  - 한국지식재산보호원
  - 한국철도공사 (용역)
  - 한국해양조사협회
--------------------------------------------------
■ 대학/교육기관(education) (총 11개)
  - 경희대학교
  - 고려대학교
  - 광주과학기술원
  - 남서울대학교
  - 대전대학교
  - 서영대학교 산학협력단
  - 서울시립대학교
  - 을지대학교
  - 전북대학교
  - 조선대학교
  - 한영대학
--------------------------------------------------
■ 민간기업(company) (총 1개)
  - 케빈랩 주식회사
-------------------------

In [46]:
df.loc[is_hwp, "기관 섹터"].value_counts()  # pdf : 연구기관 1, 지자체 1, 교육 2

기관 섹터
공기업/공공기관(public)        55
대학/교육기관(education)      10
연구기관(research)          10
지자체(local)               9
재단/협회/비영리(nonprofit)     9
중앙행정기관(central)          2
민간기업(company)            1
Name: count, dtype: int64

In [47]:
is_pdf = df["파일명"].str.lower().str.endswith(".pdf")
df.loc[is_pdf, "기관 섹터"].value_counts()

기관 섹터
대학/교육기관(education)    2
지자체(local)            1
연구기관(research)        1
Name: count, dtype: int64

In [48]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      100 non-null    str    
 1   공고 차수      82 non-null     float64
 2   사업명        100 non-null    str    
 3   사업 금액      100 non-null    float64
 4   발주 기관      100 non-null    str    
 5   공개 일자      100 non-null    str    
 6   입찰 참여 시작일  100 non-null    str    
 7   입찰 참여 마감일  93 non-null     str    
 8   사업 요약      100 non-null    str    
 9   파일형식       100 non-null    str    
 10  파일명        100 non-null    str    
 11  텍스트        100 non-null    str    
 12  텍스트길이      7 non-null      float64
 13  기관 섹터      100 non-null    str    
dtypes: float64(3), str(11)
memory usage: 1.5 MB


## 텍스트 정제

### 공통 정제 함수

In [49]:
def clean_text_common(text):
    """
    최소 전처리 공통 함수
    - 의미/구조를 바꿀 수 있는 공격적 정제는 하지 않음
    - 모든 섹터에 공통으로 안전한 정리만 수행
    """
    if pd.isna(text):
        return text

    text = str(text)

    # 1) 줄바꿈 통일
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # 2) 탭 정리
    text = text.replace("\t", " ")

    # 3) 명백히 깨진 일부 특수문자 제거
    text = re.sub(r"[↸ᬄὩ⇟]", " ", text)

    # 4) 제어문자 제거
    text = re.sub(r"[\x00-\x08\x0b-\x1f\x7f]", " ", text)

    # 5) 줄 내부의 연속 공백만 축소
    #    줄바꿈 구조는 건드리지 않음
    text = re.sub(r"[ ]{2,}", " ", text)

    # 6) 줄 양끝 공백 제거
    text = "\n".join(line.strip() for line in text.split("\n"))

    # 7) 과도한 빈 줄만 축소
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [50]:
df["공통정제텍스트"] = df["텍스트"].apply(clean_text_common)

In [51]:
len(df["공통정제텍스트"])

100

In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      100 non-null    str    
 1   공고 차수      82 non-null     float64
 2   사업명        100 non-null    str    
 3   사업 금액      100 non-null    float64
 4   발주 기관      100 non-null    str    
 5   공개 일자      100 non-null    str    
 6   입찰 참여 시작일  100 non-null    str    
 7   입찰 참여 마감일  93 non-null     str    
 8   사업 요약      100 non-null    str    
 9   파일형식       100 non-null    str    
 10  파일명        100 non-null    str    
 11  텍스트        100 non-null    str    
 12  텍스트길이      7 non-null      float64
 13  기관 섹터      100 non-null    str    
 14  공통정제텍스트    100 non-null    str    
dtypes: float64(3), str(12)
memory usage: 2.9 MB


### 섹터별 정제함수

In [53]:
# clean_text_common적용한 공통정제텍스트와 동일한 정제테스트 컬럼만들고,
# 정제테스트 컬럼에 기관섹터별 함수(ex.clean_text_public) 적용한 텍스트를 덮어씀,
# 공통정제텍스트 컬럼은 그대로 보존 //
# 정제텍스트 컬럼은 처음엔 공통정제텍스트 복사본으로 시작하는 것,
# 그 다음 public, rnd, local ... 함수가 해당 행만 덮어쓰면서 누적 반영 //

df["정제텍스트"] = df["공통정제텍스트"].copy()

#### 1. 공기업/공공기관(public)

In [55]:
def clean_text_public(text):
    """
    공기업/공공기관(public) 전용 보수적 정제

    원칙
    - 연락처/이메일/URL은 보호
    - 제목형 띄어쓰기만 제한적으로 정리
    - 2줄짜리 짧은 표제/장제목 일부 결합
    - 목차형 줄 끝 페이지번호 제거
    - 확실한 OCR/이미지 잔여물만 제거
    - 본문 의미 훼손 가능성이 있는 과도한 치환은 지양
    """

    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",                         # 063-716-2787
        r"^(?:TEL|FAX)\s*:\s*\d{2,4}-\d{2,4}-\d{3,4}$",       # TEL: 042-615-5670
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",  # email
        r"^https?://\S+$",                                    # url
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)

    # 자주 보이는 표기 통일
    text = text.replace("㈜", "(주)")
    text = text.replace("（", "(").replace("）", ")")

    # --------------------------------------------------
    # 2) 줄 단위 보호 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX)\s*:\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 3) 2줄짜리 짧은 표제 결합
    # --------------------------------------------------
    merge_pairs = {
        ("목", "차"): "목차",
        ("순", "서"): "순서",
        ("담", "당"): "담당",
        ("소", "속"): "소속",
        ("성", "명"): "성명",
        ("직", "위"): "직위",
        ("전", "화"): "전화",
        ("부", "서"): "부서",
        ("부서", "명"): "부서명",
        ("전화", "번호"): "전화번호",
        ("주", "관"): "주관",
        ("기", "관"): "기관",
        ("주관", "기관"): "주관기관",
        ("사", "업"): "사업",
        ("개", "요"): "개요",
        ("붙", "임"): "붙임",
        ("별", "표"): "별표",
        ("별", "지"): "별지",
        ("서", "식"): "서식",
    }

    roman_only_pattern = re.compile(r"^[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+[\.．]?$")

    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    while i < len(lines):
        cur = lines[i]

        # 빈 줄은 그대로
        if not cur:
            merged_lines.append("")
            i += 1
            continue

        if i + 1 < len(lines):
            nxt = lines[i + 1].strip()

            # 3-1) 짧은 2줄 표제 결합
            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # 3-2) 로마숫자 단독 줄 + 다음 줄 제목 결합
            # 예: "Ⅰ" + "사업 개요" -> "Ⅰ. 사업 개요"
            if roman_only_pattern.fullmatch(cur) and nxt:
                merged_lines.append(cur.rstrip(".．") + ". " + nxt)
                i += 2
                continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 4) 보수적으로 붙여도 되는 짧은 표제 사전
    # --------------------------------------------------
    header_compact_map = {
        "제 안 요 청 서": "제안요청서",
        "과 업 지 시 서": "과업지시서",
        "사 업 명": "사업명",
        "사 업 비": "사업비",
        "사 업 개 요": "사업개요",
        "사 업 범 위": "사업범위",
        "사 업 기 간": "사업기간",
        "추 진 배 경": "추진배경",
        "추 진 방 안": "추진방안",
        "추 진 목 표": "추진목표",
        "추 진 일 정": "추진일정",
        "주 관 기 관": "주관기관",
        "수 요 기 관": "수요기관",
        "부 서 명": "부서명",
        "전 화 번 호": "전화번호",
        "담 당 자": "담당자",
        "목 차": "목차",
        "순 서": "순서",
        "일 반 사 항": "일반사항",
        "기 대 효 과": "기대효과",
        "제 안 안 내 사 항": "제안 안내사항",
        "제 안 서 작 성 요 령": "제안서 작성요령",
        "제 안 요 청 내 용": "제안요청 내용",
        "정 보 시 스 템 현 황": "정보시스템 현황",
    }

    # --------------------------------------------------
    # 5) 목차형 패턴
    # --------------------------------------------------
    toc_head_pattern = re.compile(
        r"""^(
            \[.*\]|
            <.*>|
            [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+[\.．]?\s*|
            제?\d+장\s+|
            \d+(\.\d+)*[\.\)]\s+|
            [가나다라마바사아자차카타파하][\.\)]\s+|
            (별지|별표|붙임|서식|첨부서식)\s*\d*\.?\s*
        )""",
        re.VERBOSE
    )

    roman_prefix_pattern = re.compile(r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+[\.．]?\s*)(.+)$")

    # 번호 항목 시작부에서만 보수적으로 붙일 제목들
    numbered_header_terms = [
        "사 업 명",
        "사 업 비",
        "사 업 기 간",
        "사 업 범 위",
        "사 업 개 요",
        "추 진 배 경",
        "추 진 방 안",
        "추 진 목 표",
        "추 진 일 정",
        "사 업 내 용",
        "사 업 예 산",
        "기 대 효 과",
    ]
    numbered_header_pattern = re.compile(
        r"^(\d+\.\s*)(" + "|".join(re.escape(x) for x in numbered_header_terms) + r")(\s*[:：]?\s*)(.*)$"
    )
    new_lines = []

    for line in merged_lines:
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        # 보호 라인 그대로 유지
        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        # ----------------------------------------------
        # 5-1) 확실한 OCR/이미지 잔여물 제거
        # ----------------------------------------------
        if re.fullmatch(r"(세로|가로)\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        # 기호만 있는 짧은 줄 제거
        if re.fullmatch(r"[0Oo○◦·•\-=]{1,4}", s):
            continue

        # 숫자만 있는 아주 짧은 줄 제거 (고립 페이지번호 가능성)
        if re.fullmatch(r"\d{1,2}", s):
            continue

        # ----------------------------------------------
        # 5-2) 표제 사전 치환
        # ----------------------------------------------
        if s in header_compact_map:
            s = header_compact_map[s]

        # ----------------------------------------------
        # 5-3) 번호 항목 시작부의 제목형 띄어쓰기 정리
        # 예: "1. 사 업 명 :" -> "1. 사업명 :"
        # ----------------------------------------------
        s = numbered_header_pattern.sub(
            lambda m: m.group(1) + m.group(2).replace(" ", "") + m.group(3) + m.group(4),
            s
        )

        # ----------------------------------------------
        # 5-4) 짧은 제목형 띄어쓰기 정리
        # 너무 일반적인 본문에는 적용하지 않음
        # ----------------------------------------------
        if (
            len(s) <= 20
            and ":" not in s
            and not re.search(r"\d{2,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,15}[가-힣A-Za-z]", s)
        ):
            s = s.replace(" ", "")

        # ----------------------------------------------
        # 5-5) 로마숫자 장/절 제목 정리
        # 예: "Ⅰ. 사 업 개 요" -> "Ⅰ. 사업 개요"
        # ----------------------------------------------
        m = roman_prefix_pattern.match(s)
        if m:
            prefix, body = m.group(1), m.group(2).strip()

            if (
                len(body) <= 30
                and not re.search(r"\d{2,}", body)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,20}[가-힣A-Za-z]", body)
            ):
                compact = body.replace(" ", "")
                spaced_title_map = {
                    "사업개요": "사업 개요",
                    "사업추진방안": "사업추진 방안",
                    "정보시스템현황": "정보시스템 현황",
                    "제안요청내용": "제안요청 내용",
                    "제안안내사항": "제안 안내사항",
                    "제안서작성요령": "제안서 작성요령",
                    "운영환경": "운영 환경",
                }
                body = spaced_title_map.get(compact, compact)
                s = prefix + body

        # ----------------------------------------------
        # 5-6) 괄호 안 제목형 띄어쓰기
        # ----------------------------------------------
        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 20
                and not re.search(r"\d{2,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,15}[가-힣A-Za-z]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z]\s){1,15}[가-힣A-Za-z])\)",
            fix_spaced_korean_in_parens,
            s
        )

        # ----------------------------------------------
        # 5-7) 목차형 줄 끝 페이지번호 제거
        # 예: "- 7", "- - 62", "··· 12"
        # ----------------------------------------------
        if toc_head_pattern.match(s):
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·•‧\.\…]{2,}\s*\d{1,3}\s*$", "", s)

        # 목차 아닌 줄에서도 아주 전형적인 짧은 제목줄의 끝 페이지번호만 제거
        if (
            len(s) <= 60
            and ":" not in s
            and re.search(r"\s-\s\d{1,3}$", s)
        ):
            s = re.sub(r"\s-\s\d{1,3}$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 6) 후처리
    # --------------------------------------------------
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

#### 2. 연구기관(research)

In [56]:
def clean_text_research(text):
    """
    연구기관(research) 전용 보수적 정제

    원칙
    - 연락처/이메일/URL은 보호
    - 제목형 띄어쓰기만 제한적으로 정리
    - 2줄짜리 짧은 표제/장제목 일부 결합
    - 목차형 줄 끝 페이지번호 제거
    - 확실한 OCR/이미지 잔여물만 제거
    - 본문 의미 훼손 가능성이 있는 과도한 치환은 지양
    - 표지/기관명/사업명 같은 핵심 메타정보는 보존
    """

    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^(?:TEL|FAX)\s*:\s*\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        r"^https?://\S+$",
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)

    text = text.replace("㈜", "(주)")
    text = text.replace("（", "(").replace("）", ")")

    # --------------------------------------------------
    # 2) 문서 앞부분 OCR 깨짐 헤더 제한 제거
    #    너무 공격적으로 지우지 않도록 약하게 조정
    #    - 첫 줄이 매우 지저분할 때만
    #    - 제안요청서/사업명/과업명/기관명이 바로 뒤에 나오는 경우만
    # --------------------------------------------------
    front_noise_pattern = re.compile(
        r"^([^\n]{0,120})\n",
        re.MULTILINE
    )
    m = front_noise_pattern.match(text)
    if m:
        first_line = m.group(1).strip()
        rest = text[m.end():]

        looks_noisy = (
            len(first_line) >= 12
            and (
                len(re.findall(r"[가-힣A-Za-z0-9]", first_line)) / max(len(first_line), 1) < 0.7
                or len(re.findall(r"\b[가-힣A-Za-z0-9]\b", first_line)) >= 6
                or bool(re.search(r"[쌀뀀툀턀럀]+", first_line))
            )
        )

        next_has_anchor = bool(re.search(
            r"(제\s*안\s*요\s*청\s*서|제안요청서|사업명|과업명|발주기관|주관기관|20\d{2}\.\s*\d{1,2}\.)",
            rest[:300]
        ))

        if looks_noisy and next_has_anchor:
            text = rest.lstrip()

    # --------------------------------------------------
    # 3) 보호 줄 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX)\s*:\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 4) 2줄 제목 결합
    # --------------------------------------------------
    merge_pairs = {
        ("목", "차"): "목차",
        ("순", "서"): "순서",
        ("담", "당"): "담당",
        ("소", "속"): "소속",
        ("성", "명"): "성명",
        ("직", "위"): "직위",
        ("전", "화"): "전화",
        ("부", "서"): "부서",
        ("부서", "명"): "부서명",
        ("전화", "번호"): "전화번호",
        ("사", "업"): "사업",
        ("과", "업"): "과업",
        ("개", "요"): "개요",
        ("현", "황"): "현황",
        ("안", "내"): "안내",
        ("붙", "임"): "붙임",
        ("별", "표"): "별표",
        ("별", "지"): "별지",
        ("서", "식"): "서식",
        ("주", "관"): "주관",
        ("기", "관"): "기관",
        ("주관", "기관"): "주관기관",
        ("발주", "기관"): "발주기관",
    }

    roman_only_pattern = re.compile(
    r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|(?=[IVXLC]+$)I|II|III|IV|V|VI|VII|VIII|IX|X|XI|XII|XIII|XIV|XV)[\.．]?$"
    )

    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    while i < len(lines):
        cur = lines[i]

        if not cur:
            merged_lines.append("")
            i += 1
            continue

        if i + 1 < len(lines):
            nxt = lines[i + 1].strip()

            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # 예: "Ⅰ" + "사업 개요" -> "Ⅰ. 사업 개요"
            if roman_only_pattern.fullmatch(cur):
                roman_map = {
                    "I": "Ⅰ", "II": "Ⅱ", "III": "Ⅲ", "IV": "Ⅳ", "V": "Ⅴ",
                    "VI": "Ⅵ", "VII": "Ⅶ", "VIII": "Ⅷ", "IX": "Ⅸ", "X": "Ⅹ"
                }
                cur_norm = roman_map.get(cur.rstrip(".．"), cur.rstrip(".．"))

                # 바로 다음 줄이 비어 있으면 한 줄 더 본다
                if nxt:
                    merged_lines.append(cur_norm + ". " + nxt)
                    i += 2
                    continue
                elif i + 2 < len(lines):
                    nxt2 = lines[i + 2].strip()
                    if nxt2:
                        merged_lines.append(cur_norm + ". " + nxt2)
                        i += 3
                        continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 5) 표제 사전
    # --------------------------------------------------
    header_compact_map = {
        "제 안 요 청 서": "제안요청서",
        "사 업 명": "사업명",
        "과 업 명": "과업명",
        "발 주 기 관": "발주기관",
        "주 관 기 관": "주관기관",
        "부 서 명": "부서명",
        "전 화 번 호": "전화번호",
        "담 당 자": "담당자",
        "목 차": "목차",
        "< 목 차 >": "<목차>",
        "사 업 개 요": "사업개요",
        "과 업 개 요": "과업개요",
        "사 업 범 위": "사업범위",
        "과 업 범 위": "과업범위",
        "사 업 기 간": "사업기간",
        "과 업 기 간": "과업기간",
        "추 진 배 경": "추진배경",
        "추 진 목 적": "추진목적",
        "추 진 방 안": "추진방안",
        "추 진 계 획": "추진계획",
        "추 진 일 정": "추진일정",
        "기 대 효 과": "기대효과",
        "제 안 요 청 내 용": "제안요청 내용",
        "제 안 안 내 사 항": "제안 안내사항",
        "제 안 서 작 성 요 령": "제안서 작성요령",
        "정 보 시 스 템 현 황": "정보시스템 현황",
        "대 상 업 무 현 황": "대상 업무 현황",
    }

    toc_head_pattern = re.compile(
        r"""^(
            \[.*\]|
            <.*>|
            [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*|
            제?\d+장\s+|
            \d+(\.\d+)*[\.\)]\s+|
            [가나다라마바사아자차카타파하][\.\)]\s+|
            (별지|별표|붙임|서식|첨부서식|별첨|부록|참고자료)\s*\d*\.?\s*
        )""",
        re.VERBOSE
    )

    roman_prefix_pattern = re.compile(
        r"^((?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|(?=[IVXLC]+$)[IVXLC]+)[\.．]?\s*)(.+)$"
    )

    # 번호 항목 시작부에서만 붙일 제목
    numbered_header_terms = [
        "사 업 명", "과 업 명",
        "사 업 비", "사 업 예 산",
        "사 업 기 간", "과 업 기 간",
        "사 업 범 위", "과 업 범 위",
        "사 업 개 요", "과 업 개 요",
        "추 진 배 경", "추 진 목 적",
        "추 진 방 안", "추 진 계 획",
        "추 진 목 표", "추 진 일 정",
        "사 업 내 용", "과 업 내 용",
        "기 대 효 과",
    ]
    numbered_header_pattern = re.compile(
        r"^(\d+\.\s*)(" + "|".join(re.escape(x) for x in numbered_header_terms) + r")(\s*[:：]?\s*)(.*)$"
    )

    # 표지 핵심어 보호
    cover_meta_keywords = [
        "제안요청서", "제 안 요 청 서", "제안요청서(RFP)", "RFP",
        "사업명", "과업명", "발주기관", "주관기관",
        "한국생산기술연구원", "국방과학연구소", "충북연구원",
        "광주연구원", "한국원자력연구원", "한국한의학연구원",
        "한국수자원조사기술원"
    ]

    new_lines = []

    for idx, line in enumerate(merged_lines):
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        # 보호 라인 그대로 유지
        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        # 표지 핵심 정보는 삭제/과도치환 방지
        if any(k in s for k in cover_meta_keywords):
            if s in header_compact_map:
                s = header_compact_map[s]
            new_lines.append(s)
            continue

        # ----------------------------------------------
        # 5-1) 확실한 OCR/이미지 잔여물 제거
        # ----------------------------------------------
        if re.fullmatch(r"(세로|가로)\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        if re.fullmatch(r"[·•○◦\-_=~]{1,6}", s):
            continue

        # 숫자 단독 줄 제거는 더 보수적으로:
        # 문서 첫머리/표/절번호 손상 막기 위해 1자리만 제거
        if re.fullmatch(r"\d", s):
            continue

        # ----------------------------------------------
        # 5-2) 이상한 목차 노이즈 문자 보정
        # 예: "1. h 사업 개요" -> "1. 사업 개요"
        # ----------------------------------------------
        s = re.sub(r"^(\d+\.\s*)[hH]\s+", r"\1", s)

        # ----------------------------------------------
        # 5-3) 표제 사전 치환
        # ----------------------------------------------
        if s in header_compact_map:
            s = header_compact_map[s]

        # ----------------------------------------------
        # 5-4) 번호 항목 시작부 제목형 정리
        # 예: "1. 사 업 명 : ..." -> "1. 사업명 : ..."
        # ----------------------------------------------
        s = numbered_header_pattern.sub(
            lambda m: m.group(1) + m.group(2).replace(" ", "") + m.group(3) + m.group(4),
            s
        )

        # ----------------------------------------------
        # 5-5) 짧은 제목형 띄어쓰기 정리
        # ----------------------------------------------
        if (
            len(s) <= 35
            and ":" not in s
            and not re.search(r"\d{3,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z]", s)
        ):
            s = s.replace(" ", "")

        # ----------------------------------------------
        # 5-6) 로마숫자 장/절 제목 정리
        # 예: "Ⅰ. 사 업 개 요" -> "Ⅰ. 사업 개요"
        # ----------------------------------------------
        m = roman_prefix_pattern.match(s)
        if m:
            prefix, body = m.group(1), m.group(2).strip()

            if (
                len(body) <= 35
                and not re.search(r"\d{3,}", body)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z]", body)
            ):
                compact = body.replace(" ", "")
                spaced_title_map = {
                    "사업개요": "사업 개요",
                    "과업개요": "과업 개요",
                    "사업추진방안": "사업 추진방안",
                    "사업추진계획": "사업 추진 계획",
                    "정보시스템현황": "정보시스템 현황",
                    "대상업무현황": "대상 업무 현황",
                    "제안요청내용": "제안요청 내용",
                    "제안안내사항": "제안 안내사항",
                    "제안서작성요령": "제안서 작성요령",
                }
                body = spaced_title_map.get(compact, compact)
                s = prefix + body

        # ----------------------------------------------
        # 5-7) 괄호 안 제목형 띄어쓰기
        # ----------------------------------------------
        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 35
                and not re.search(r"\d{3,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z])\)",
            fix_spaced_korean_in_parens,
            s
        )

        # ----------------------------------------------
        # 5-8) 목차형 줄 끝 페이지번호 제거
        # ----------------------------------------------
        if toc_head_pattern.match(s):
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·•‧\.\…]{2,}\s*\d{1,3}\s*$", "", s)

        # 짧은 제목줄 끝 페이지번호 제거
        if (
            len(s) <= 70
            and ":" not in s
            and re.search(r"\s-\s\d{1,3}$", s)
        ):
            s = re.sub(r"\s-\s\d{1,3}$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 6) 후처리
    # --------------------------------------------------
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

#### 3. 대학/교육기관(education)

In [57]:
def clean_text_education(text):
    """
    대학/교육기관(education) 전용 보수적 정제

    원칙
    - 연락처/이메일/URL은 보호
    - 제목형 띄어쓰기만 제한적으로 정리
    - 장/절 제목이 줄바꿈으로 분리된 경우 일부 결합
    - 목차형 줄 끝 페이지번호 제거
    - 확실한 OCR/이미지 잔여물만 제거
    - 본문/표 의미 훼손 가능성이 있는 과도한 삭제는 지양
    """
    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^(?:TEL|FAX|Tel\.?|Fax\.?|전화)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        r"^https?://\S+$",
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)
    text = text.replace("㈜", "(주)")
    text = text.replace("（", "(").replace("）", ")")
    text = re.sub(r"[–—−]", "-", text)

    # --------------------------------------------------
    # 2) 보호 줄 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX|Tel\.?|Fax\.?|전화)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 3) 줄 단위 제목 결합
    # --------------------------------------------------
    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    merge_pairs = {
        ("목", "차"): "목차",
        ("개", "요"): "개요",
        ("현", "황"): "현황",
        ("붙", "임"): "붙임",
        ("별", "표"): "별표",
        ("별", "지"): "별지",
        ("서", "식"): "서식",
        ("안", "내"): "안내",
        ("사", "업"): "사업",
        ("제안요청", "사항"): "제안요청 사항",
        ("제안안내", "사항"): "제안안내 사항",
    }

    roman_only_pattern = re.compile(
        r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|[IVXLC]+)[\.．]?$"
    )

    # 로마숫자 뒤에 붙이기 좋은 교육기관형 장 제목
    edu_section_title_pattern = re.compile(
        r"(?:[가-힣A-Za-z]\s){0,25}[가-힣A-Za-z]"
    )

    spaced_title_map = {
        "사업안내": "사업 안내",
        "사업개요": "사업 개요",
        "구축방안": "구축 방안",
        "제안요청내용": "제안요청 내용",
        "제안요청사항": "제안요청 사항",
        "제안안내사항": "제안안내 사항",
        "참조자료": "참조자료",
        "사업추진방안": "사업 추진방안",
        "사업추진방향": "사업 추진방향",
        "사업배경및목적": "사업배경 및 목적",
    }

    while i < len(lines):
        cur = lines[i]

        if not cur:
            merged_lines.append("")
            i += 1
            continue

        if i + 1 < len(lines):
            nxt = lines[i + 1].strip()

            # 보호 줄은 건드리지 않음
            if protected_line_pattern.search(cur) or protected_line_pattern.search(nxt):
                merged_lines.append(cur)
                i += 1
                continue

            # 1) 미리 정의한 짧은 표제 결합
            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # 2) 로마숫자 장 제목: Ⅰ / 사업 안내 -> Ⅰ. 사업 안내
            if roman_only_pattern.fullmatch(cur):
                roman_map = {
                    "I": "Ⅰ", "II": "Ⅱ", "III": "Ⅲ", "IV": "Ⅳ", "V": "Ⅴ",
                    "VI": "Ⅵ", "VII": "Ⅶ", "VIII": "Ⅷ", "IX": "Ⅸ", "X": "Ⅹ"
                }
                cur_norm = roman_map.get(cur.rstrip(".．"), cur.rstrip(".．"))

                def normalize_section_title(x):
                    body = x.replace(" ", "")
                    return spaced_title_map.get(body, x.replace(" ", ""))

                # 바로 다음 줄에 제목이 있으면 결합
                if nxt and edu_section_title_pattern.fullmatch(nxt):
                    merged_lines.append(f"{cur_norm}. {normalize_section_title(nxt)}")
                    i += 2
                    continue

                # 빈 줄 하나를 건너뛰고 그 다음 줄에 제목이 있으면 결합
                if not nxt and i + 2 < len(lines):
                    nxt2 = lines[i + 2].strip()
                    if nxt2 and edu_section_title_pattern.fullmatch(nxt2):
                        merged_lines.append(f"{cur_norm}. {normalize_section_title(nxt2)}")
                        i += 3
                        continue

            # 3) 숫자 절 제목: 1 / 사업개요 -> 1. 사업개요
            if re.fullmatch(r"\d{1,2}", cur):
                if nxt and len(nxt) <= 40 and ":" not in nxt:
                    merged_lines.append(f"{cur}. {nxt}")
                    i += 2
                    continue
                elif not nxt and i + 2 < len(lines):
                    nxt2 = lines[i + 2].strip()
                    if nxt2 and len(nxt2) <= 40 and ":" not in nxt2:
                        merged_lines.append(f"{cur}. {nxt2}")
                        i += 3
                        continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 4) 줄 단위 세부 정제
    # --------------------------------------------------
    new_lines = []

    for idx, line in enumerate(merged_lines):
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        # 보호 줄 유지
        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        # ----------------------------------------------
        # 4-1) 확실한 OCR/이미지 잔여물 제거
        # ----------------------------------------------
        if re.fullmatch(r"(세로|가로)\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        if re.fullmatch(r"[·•○◦\-_=~]{1,6}", s):
            continue

        # 단독 숫자 줄 제거
        if re.fullmatch(r"\d{1,2}", s):
            continue

        # 고립 노이즈 제거
        if s in {"-", "l", "I"}:
            continue

        # 중간에 튀어나온 목차 제거
        # 상단부 첫 목차는 남기고, 문서 중간 이후 반복되는 목차류만 제거
        if idx > 12 and s in {"목차", "<목차>", "< 목 차 >"}:
            continue

        # ----------------------------------------------
        # 4-2) 표제형 띄어쓰기 제한적 정리
        # ----------------------------------------------
        if (
            len(s) <= 35
            and ":" not in s
            and not re.search(r"\d{3,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z]", s)
        ):
            s = s.replace(" ", "")

        # 예: "Ⅰ. 사 업 개 요" -> "Ⅰ. 사업개요"
        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*)([가-힣A-Za-z](?:\s[가-힣A-Za-z]){1,25})$",
            lambda m: m.group(1) + m.group(2).replace(" ", ""),
            s
        )

        # "Ⅰ 개요" -> "Ⅰ. 개요"
        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+)\s+([가-힣A-Za-z].+)$",
            r"\1. \2",
            s
        )

        # "1 사업개요" -> "1. 사업개요"
        s = re.sub(
            r"^(\d{1,2})\s+([가-힣A-Za-z].+)$",
            r"\1. \2",
            s
        )

        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 35
                and not re.search(r"\d{3,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z]\s){1,25}[가-힣A-Za-z])\)",
            fix_spaced_korean_in_parens,
            s
        )

        # ----------------------------------------------
        # 4-3) 목차형 줄 끝 페이지 번호 제거
        # ----------------------------------------------
        is_toc_like = bool(re.match(
            r"""^(
                \[.*\]|
                <.*>|
                [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*|
                제?\d+장\s+|
                \d+(\.\d+)*[\.\)]\s+|
                [가나다라마바사아자차카타파하][\.\)]\s+|
                (별지|별표|붙임|서식|참조)\s*\d*|
                (사업명|사업개요|사업범위|추진배경|기대효과|제안서|제안요청|제안안내|기타사항)
            )""",
            s,
            re.VERBOSE
        ))

        if is_toc_like:
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·\.…]{3,}\s*\d{1,3}\s*$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 5) 후처리
    # --------------------------------------------------
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

#### 4. 지자체(local)

In [58]:
def clean_text_local(text):
    """
    지자체(local) 전용 보수적 정제

    원칙
    - 본문 핵심 정보 보존 우선
    - 연락처/이메일/URL은 보호
    - 제목형 띄어쓰기만 제한적으로 정리
    - 2줄짜리 짧은 표제/장제목 일부 결합
    - 목차형 줄 끝 페이지번호 제거
    - 확실한 OCR/파싱 잔여물만 제거
    - 본문/표 의미 훼손 가능성이 있는 과도한 삭제는 지양
    """
    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^(?:TEL|FAX)\s*:\s*\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        r"^https?://\S+$",
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)
    text = text.replace("㈜", "(주)")

    # --------------------------------------------------
    # 2) 줄 보호 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 3) 줄 결합
    # --------------------------------------------------
    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    merge_pairs = {
        ("목", "차"): "목차",
        ("순", "서 -"): "순서",
        ("순", "서"): "순서",
        ("개", "요"): "개요",
        ("현", "황"): "현황",
        ("붙", "임"): "붙임",
        ("별", "표"): "별표",
        ("별", "지"): "별지",
        ("서", "식"): "서식",
        ("안", "내"): "안내",
        ("사", "업"): "사업",
    }

    while i < len(lines):
        cur = lines[i]

        if i + 1 < len(lines):
            nxt = lines[i + 1]

            if protected_line_pattern.search(cur) or protected_line_pattern.search(nxt):
                merged_lines.append(cur)
                i += 1
                continue

            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # Ⅰ / 사업개요  -> Ⅰ 사업개요
            if re.fullmatch(r"[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+", cur) and re.fullmatch(
                r"(?:[가-힣A-Za-z]\s){0,25}[가-힣A-Za-z]", nxt
            ):
                merged_lines.append(f"{cur} {nxt.replace(' ', '')}")
                i += 2
                continue

            # 1 / 사업일반 -> 1 사업일반
            if re.fullmatch(r"\d{1,2}(?:\.\d{1,2})?", cur) and len(nxt) <= 35 and ":" not in nxt:
                merged_lines.append(f"{cur} {nxt}")
                i += 2
                continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 4) 줄 단위 정제
    # --------------------------------------------------
    new_lines = []

    for line in merged_lines:
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        # 보호 줄 유지
        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        # ----------------------------------------------
        # 4-1) 확실한 OCR/파싱 잔여물 제거
        # ----------------------------------------------
        if re.fullmatch(r"세로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue
        if re.fullmatch(r"가로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        # 단독 노이즈 기호
        if s in {"ㅣ", "|", "ᚺ"}:
            continue

        # 의미 없는 기호성 단독 줄만 제거
        if re.fullmatch(r"[·•○◦\-_=~]{1,6}", s):
            continue

        # 숫자만 있는 매우 짧은 줄 제거 (고립 페이지번호 가능성)
        if re.fullmatch(r"\d{1,2}", s):
            continue

        # '순 서 -', '순서 -' 같은 줄 정리
        if re.fullmatch(r"순\s*서\s*-\s*", s):
            s = "순서"

        # 줄 끝에 붙은 이상 문자 제거
        s = re.sub(r"\s*[ㅣᚺ]+\s*$", "", s)

        # ----------------------------------------------
        # 4-2) 제목형 띄어쓰기 제한 정리
        # ----------------------------------------------
        # 예: "제 안 요 청 서" -> "제안요청서"
        if (
            len(s) <= 40
            and ":" not in s
            and not re.search(r"\d{3,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,30}[가-힣A-Za-z]", s)
        ):
            s = s.replace(" ", "")

        # 예: "Ⅰ. 사 업 개 요" -> "Ⅰ. 사업개요"
        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*)([가-힣A-Za-z](?:\s[가-힣A-Za-z]){1,30})$",
            lambda m: m.group(1) + m.group(2).replace(" ", ""),
            s
        )

        # 괄호 안 제목형 띄어쓰기
        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 40
                and not re.search(r"\d{3,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,30}[가-힣A-Za-z]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z]\s){1,30}[가-힣A-Za-z])\)",
            fix_spaced_korean_in_parens,
            s
        )

        # ----------------------------------------------
        # 4-3) 목차형 줄 끝 페이지번호 제거
        # ----------------------------------------------
        is_toc_like = bool(re.match(
            r"""^(
                \[.*\]|
                <.*>|
                [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*|
                제?\d+장\s+|
                \d+(\.\d+)*[\.\)]\s+|
                [가나다라마바사아자차카타파하][\.\)]\s+|
                (별지|별표|붙임|서식)\s*\d*
            )""",
            s,
            re.VERBOSE
        ))

        if is_toc_like:
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·\.…]{3,}\s*\d{1,3}\s*$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 5) 빈 줄 정리
    # --------------------------------------------------
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

#### 5. 재단/협회/비영리(nonprofit)

In [59]:
def clean_text_nonprofit(text):
    """
    재단/협회/비영리(nonprofit) 전용 보수적 정제
    """
    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^(?:TEL|FAX)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        r"^https?://\S+$",
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)
    text = text.replace("㈜", "(주)")
    text = text.replace("（", "(").replace("）", ")")
    text = re.sub(r"[–—−]", "-", text)

    # --------------------------------------------------
    # 2) 줄 보호 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 3) 줄 결합
    # --------------------------------------------------
    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    merge_pairs = {
        ("목", "차"): "목차",
        ("순", "서"): "순서",
        ("순", "서 -"): "순서",
        ("개", "요"): "개요",
        ("현", "황"): "현황",
        ("붙", "임"): "붙임",
        ("별", "표"): "별표",
        ("별", "지"): "별지",
        ("서", "식"): "서식",
        ("안", "내"): "안내",
        ("사", "업"): "사업",
        ("과", "업"): "과업",
    }

    roman_only_pattern = re.compile(
        r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|[IVXLC]+)[\.．]?$"
    )

    while i < len(lines):
        cur = lines[i]

        if not cur:
            merged_lines.append("")
            i += 1
            continue

        if i + 1 < len(lines):
            nxt = lines[i + 1]

            if protected_line_pattern.search(cur) or protected_line_pattern.search(nxt):
                merged_lines.append(cur)
                i += 1
                continue

            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # 로마숫자 장 제목 결합
            if roman_only_pattern.fullmatch(cur):
                roman_map = {
                    "I": "Ⅰ", "II": "Ⅱ", "III": "Ⅲ", "IV": "Ⅳ", "V": "Ⅴ",
                    "VI": "Ⅵ", "VII": "Ⅶ", "VIII": "Ⅷ", "IX": "Ⅸ", "X": "Ⅹ"
                }
                cur_norm = roman_map.get(cur.rstrip(".．"), cur.rstrip(".．"))

                if nxt and len(nxt) <= 35 and ":" not in nxt:
                    merged_lines.append(f"{cur_norm}. {nxt}")
                    i += 2
                    continue
                elif i + 2 < len(lines):
                    nxt2 = lines[i + 2].strip()
                    if nxt2 and len(nxt2) <= 35 and ":" not in nxt2:
                        merged_lines.append(f"{cur_norm}. {nxt2}")
                        i += 3
                        continue

            # 숫자 절 제목 결합
            # 본문의 "1 / 사업목적"만 허용하고,
            # BIFF 같은 목차 페이지번호 줄은 막는다.
            if re.fullmatch(r"\d{1,2}(?:\.\d{1,2})?", cur):
                prev_line = merged_lines[-1].strip() if merged_lines else ""
                prev_is_roman_or_chapter = bool(
                    re.match(r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?|제?\d+장)\b", prev_line)
                )
                prev_is_blank = (prev_line == "")

                next_is_roman_or_chapter = bool(
                    re.match(r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?|제?\d+장)\b", nxt)
                )
                next_is_numbered = bool(
                    re.match(r"^\d{1,2}(?:\.\d{1,2})?[\.\)]\s*", nxt)
                )

                # 본문 번호 결합은:
                # - 앞줄이 비어 있거나 로마숫자/장 제목일 때만
                # - 다음 줄이 짧은 제목일 때만
                # - 다음 줄이 또 다른 번호/장 제목이면 금지
                if (
                    (prev_is_blank or prev_is_roman_or_chapter)
                    and nxt
                    and len(nxt) <= 40
                    and ":" not in nxt
                    and not next_is_roman_or_chapter
                    and not next_is_numbered
                ):
                    merged_lines.append(f"{cur}. {nxt}")
                    i += 2
                    continue

                elif i + 2 < len(lines) and not nxt:
                    nxt2 = lines[i + 2].strip()
                    nxt2_is_roman_or_chapter = bool(
                        re.match(r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?|제?\d+장)\b", nxt2)
                    )
                    nxt2_is_numbered = bool(
                        re.match(r"^\d{1,2}(?:\.\d{1,2})?[\.\)]\s*", nxt2)
                    )

                    if (
                        (prev_is_blank or prev_is_roman_or_chapter)
                        and nxt2
                        and len(nxt2) <= 40
                        and ":" not in nxt2
                        and not nxt2_is_roman_or_chapter
                        and not nxt2_is_numbered
                    ):
                        merged_lines.append(f"{cur}. {nxt2}")
                        i += 3
                        continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 4) 줄 단위 정제
    # --------------------------------------------------
    new_lines = []

    for line in merged_lines:
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        if re.fullmatch(r"세로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue
        if re.fullmatch(r"가로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        if s in {"ㅣ", "|", "ᚺ"}:
            continue

        if re.fullmatch(r"신\s*-\s*", s):
            continue

        if re.fullmatch(r"[·•○◦\-_=~]{1,6}", s):
            continue

        if re.fullmatch(r"\d{1,2}", s):
            continue

        if re.fullmatch(r"순\s*서\s*-\s*", s):
            s = "순서"

        s = re.sub(r"\s*[ㅣᚺ]+\s*$", "", s)

        if (
            len(s) <= 45
            and ":" not in s
            and not re.search(r"\d{3,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z&]\s){1,35}[가-힣A-Za-z&]", s)
        ):
            s = s.replace(" ", "")

        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*)([가-힣A-Za-z&](?:\s[가-힣A-Za-z&]){1,35})$",
            lambda m: m.group(1) + m.group(2).replace(" ", ""),
            s
        )

        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+)\s+([가-힣A-Za-z].+)$",
            r"\1. \2",
            s
        )

        s = re.sub(
            r"^(\d{1,2}(?:\.\d{1,2})?)\s+([가-힣A-Za-z].+)$",
            r"\1. \2",
            s
        )

        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 45
                and not re.search(r"\d{3,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z&]\s){1,35}[가-힣A-Za-z&]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z&]\s){1,35}[가-힣A-Za-z&])\)",
            fix_spaced_korean_in_parens,
            s
        )

        is_toc_like = bool(re.match(
            r"""^(
                \*?\s*\[.*\]|
                <.*>|
                [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*|
                제?\d+장\s+|
                \d+(\.\d+)*[\.\)]\s+|
                [가나다라마바사아자차카타파하][\.\)]\s+|
                (별지|별표|붙임|서식)\s*\d*
            )""",
            s,
            re.VERBOSE
        ))

        if is_toc_like:
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·\.…]{3,}\s*\d{1,3}\s*$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 5) 후처리
    # --------------------------------------------------
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

#### 6. 중앙행정기관(central)

In [60]:
def clean_text_central(text):
    """
    중앙행정기관(central) 전용 보수적 정제

    원칙
    - 본문 핵심 정보 보존 우선
    - 연락처/이메일/URL은 보호
    - 제목형 띄어쓰기만 제한적으로 정리
    - 2줄짜리 짧은 표제/장제목 일부 결합
    - 목차형 줄 끝 페이지번호 제거
    - 확실한 OCR/파싱 잔여물만 제거
    - 본문/표 의미 훼손 가능성이 있는 과도한 삭제는 지양
    """
    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^(?:TEL|FAX|전화|팩스)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        r"^https?://\S+$",
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)
    text = text.replace("㈜", "(주)")
    text = text.replace("ῼ", " ")

    # --------------------------------------------------
    # 2) 줄 보호 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX|전화|팩스)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 3) 줄 결합
    # --------------------------------------------------
    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    merge_pairs = {
        ("목", "차"): "목차",
        ("개", "요"): "개요",
        ("현", "황"): "현황",
        ("붙", "임"): "붙임",
        ("별", "지"): "별지",
        ("별", "표"): "별표",
        ("서", "식"): "서식",
        ("안", "내"): "안내",
        ("사", "업"): "사업",
        ("요", "구"): "요구",
        ("내", "용"): "내용",
    }

    while i < len(lines):
        cur = lines[i]

        if i + 1 < len(lines):
            nxt = lines[i + 1]

            if protected_line_pattern.search(cur) or protected_line_pattern.search(nxt):
                merged_lines.append(cur)
                i += 1
                continue

            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # Ⅰ / 사 업 개 요  -> Ⅰ 사업개요
            if re.fullmatch(r"[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+", cur) and re.fullmatch(
                r"(?:[가-힣A-Za-z]\s){0,30}[가-힣A-Za-z]", nxt
            ):
                merged_lines.append(f"{cur} {nxt.replace(' ', '')}")
                i += 2
                continue

            # 1 / 사 업 일 반 -> 1 사업일반
            if re.fullmatch(r"\d{1,2}(?:\.\d{1,2})?", cur) and re.fullmatch(
                r"(?:[가-힣A-Za-z]\s){0,30}[가-힣A-Za-z]", nxt
            ):
                merged_lines.append(f"{cur} {nxt.replace(' ', '')}")
                i += 2
                continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 4) 줄 단위 정제
    # --------------------------------------------------
    new_lines = []

    for line in merged_lines:
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        # 보호 줄 유지
        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        # ----------------------------------------------
        # 4-1) 확실한 장식/노이즈만 제거
        # ----------------------------------------------
        # 예: • • • 목 차 • •  -> 목차
        if re.fullmatch(r"[•·\s]*목\s*차[•·\s]*", s):
            new_lines.append("목차")
            continue

        # 단독 장식 기호 줄
        if re.fullmatch(r"[•·○◦\-_=~]{1,10}", s):
            continue

        # pixel 잔여물
        if re.fullmatch(r"세로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue
        if re.fullmatch(r"가로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        # 숫자만 있는 매우 짧은 줄 제거 (고립 페이지번호 가능성)
        if re.fullmatch(r"\d{1,2}", s):
            continue

        # ----------------------------------------------
        # 4-2) 제목형 띄어쓰기 제한 정리
        # ----------------------------------------------
        # 예: 제 안 요 청 서 -> 제안요청서
        if (
            len(s) <= 45
            and ":" not in s
            and not re.search(r"\d{3,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,35}[가-힣A-Za-z]", s)
        ):
            s = s.replace(" ", "")

        # 예: Ⅰ. 사 업 개 요 -> Ⅰ. 사업개요
        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*)([가-힣A-Za-z](?:\s[가-힣A-Za-z]){1,35})$",
            lambda m: m.group(1) + m.group(2).replace(" ", ""),
            s
        )

        # 괄호 안 제목형 띄어쓰기
        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 45
                and not re.search(r"\d{3,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z]\s){1,35}[가-힣A-Za-z]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z]\s){1,35}[가-힣A-Za-z])\)",
            fix_spaced_korean_in_parens,
            s
        )

        # ----------------------------------------------
        # 4-3) 목차형 줄 끝 페이지번호 제거
        # ----------------------------------------------
        is_toc_like = bool(re.match(
            r"""^(
                \[.*\]|
                <.*>|
                [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*|
                제?\d+장\s+|
                \d+(\.\d+)*[\.\)]\s+|
                [가나다라마바사아자차카타파하][\.\)]\s+|
                (붙임|별지|별표|서식)\s*\d*
            )""",
            s,
            re.VERBOSE
        ))

        if is_toc_like:
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·\.…]{3,}\s*\d{1,3}\s*$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 5) 빈 줄 정리
    # --------------------------------------------------
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

#### 7. 민간기업(company)

In [61]:
def clean_text_company(text):
    """
    민간기업(company) 전용 보수적 정제

    원칙
    - 본문 핵심 정보 보존 우선
    - 연락처/이메일/URL은 보호
    - 제목형 띄어쓰기만 제한적으로 정리
    - 2줄짜리 짧은 표제/장제목 일부 결합
    - 목차형 줄 끝 페이지번호 제거
    - 목차 안의 확실한 잡음 문자만 제거
    - 요구사항 코드/기술용어/약어는 보존
    """
    if pd.isna(text):
        return text

    raw = str(text).strip()
    if not raw:
        return raw

    # --------------------------------------------------
    # 0) 입력 전체가 보호 대상이면 그대로 반환
    # --------------------------------------------------
    full_protected_patterns = [
        r"^\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^(?:TEL|FAX|전화|팩스)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}$",
        r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$",
        r"^https?://\S+$",
    ]
    for pat in full_protected_patterns:
        if re.fullmatch(pat, raw, flags=re.IGNORECASE):
            return raw

    # --------------------------------------------------
    # 1) 공통 정제
    # --------------------------------------------------
    text = clean_text_common(raw)
    text = text.replace("㈜", "(주)")
    text = text.replace("（", "(").replace("）", ")")
    text = re.sub(r"[–—−]", "-", text)

    # --------------------------------------------------
    # 2) 줄 보호 패턴
    # --------------------------------------------------
    protected_line_pattern = re.compile(
        r"("
        r"^\d{2,4}-\d{2,4}-\d{3,4}$"
        r"|(?:TEL|FAX|전화|팩스)\s*:?\s*\d{2,4}-\d{2,4}-\d{3,4}"
        r"|[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
        r"|https?://\S+"
        r")",
        re.IGNORECASE
    )

    # --------------------------------------------------
    # 3) 줄 결합
    # --------------------------------------------------
    lines = [line.strip() for line in text.split("\n")]
    merged_lines = []
    i = 0

    merge_pairs = {
        ("목", "차"): "목차",
        ("개", "요"): "개요",
        ("현", "황"): "현황",
        ("붙", "임"): "붙임",
        ("별", "표"): "별표",
        ("별", "지"): "별지",
        ("서", "식"): "서식",
        ("안", "내"): "안내",
        ("사", "업"): "사업",
        ("과", "업"): "과업",
        ("요", "구"): "요구",
        ("내", "용"): "내용",
    }

    roman_only_pattern = re.compile(
        r"^(?:[ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+|[IVXLC]+)[\.．]?$"
    )

    while i < len(lines):
        cur = lines[i]

        if not cur:
            merged_lines.append("")
            i += 1
            continue

        if i + 1 < len(lines):
            nxt = lines[i + 1]

            if protected_line_pattern.search(cur) or protected_line_pattern.search(nxt):
                merged_lines.append(cur)
                i += 1
                continue

            merged = merge_pairs.get((cur, nxt))
            if merged:
                merged_lines.append(merged)
                i += 2
                continue

            # Ⅰ / 사업개요 -> Ⅰ. 사업개요
            # Ⅰ / 빈줄 / 사업개요 -> Ⅰ. 사업개요
            if roman_only_pattern.fullmatch(cur):
                roman_map = {
                    "I": "Ⅰ", "II": "Ⅱ", "III": "Ⅲ", "IV": "Ⅳ", "V": "Ⅴ",
                    "VI": "Ⅵ", "VII": "Ⅶ", "VIII": "Ⅷ", "IX": "Ⅸ", "X": "Ⅹ"
                }
                cur_norm = roman_map.get(cur.rstrip(".．"), cur.rstrip(".．"))

                if nxt and len(nxt) <= 35 and ":" not in nxt:
                    merged_lines.append(f"{cur_norm}. {nxt}")
                    i += 2
                    continue
                elif i + 2 < len(lines):
                    nxt2 = lines[i + 2].strip()
                    if nxt2 and len(nxt2) <= 35 and ":" not in nxt2:
                        merged_lines.append(f"{cur_norm}. {nxt2}")
                        i += 3
                        continue

            # 1 / 추진배경 -> 1. 추진배경
            if re.fullmatch(r"\d{1,2}(?:\.\d{1,2})?", cur):
                if nxt and len(nxt) <= 45 and ":" not in nxt:
                    merged_lines.append(f"{cur}. {nxt}")
                    i += 2
                    continue
                elif i + 2 < len(lines) and not nxt:
                    nxt2 = lines[i + 2].strip()
                    if nxt2 and len(nxt2) <= 45 and ":" not in nxt2:
                        merged_lines.append(f"{cur}. {nxt2}")
                        i += 3
                        continue

        merged_lines.append(cur)
        i += 1

    # --------------------------------------------------
    # 4) 줄 단위 정제
    # --------------------------------------------------
    new_lines = []

    for line in merged_lines:
        s = line.strip()

        if not s:
            new_lines.append("")
            continue

        if protected_line_pattern.search(s):
            new_lines.append(s)
            continue

        # ----------------------------------------------
        # 4-1) 확실한 잔여물만 제거
        # ----------------------------------------------
        if re.fullmatch(r"세로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue
        if re.fullmatch(r"가로\s*\d+\s*pixel", s, flags=re.IGNORECASE):
            continue

        # 단독 노이즈
        if s in {"ㅣ", "|", "ᚺ", "ḿ", "⁻"}:
            continue

        # 의미 없는 장식 줄
        if re.fullmatch(r"[·•○◦\-_=~]{1,10}", s):
            continue

        # 숫자만 있는 매우 짧은 줄 제거 (페이지번호 가능성)
        if re.fullmatch(r"\d{1,2}", s):
            continue

        # 줄 끝 이상 문자 제거
        s = re.sub(r"\s*[ḿ⁻ᚺ]+\s*$", "", s)

        # ----------------------------------------------
        # 4-2) 제목형 띄어쓰기 제한 정리
        # ----------------------------------------------
        if (
            len(s) <= 50
            and ":" not in s
            and not re.search(r"\d{3,}", s)
            and re.fullmatch(r"(?:[가-힣A-Za-z&/]\s){1,40}[가-힣A-Za-z&/]", s)
        ):
            s = s.replace(" ", "")

        # 예: Ⅰ. 사 업 개 요 -> Ⅰ. 사업개요
        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*)([가-힣A-Za-z&/](?:\s[가-힣A-Za-z&/]){1,40})$",
            lambda m: m.group(1) + m.group(2).replace(" ", ""),
            s
        )

        # 예: Ⅰ 사업개요 -> Ⅰ. 사업개요
        s = re.sub(
            r"^([ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩ]+)\s+([가-힣A-Za-z].+)$",
            r"\1. \2",
            s
        )

        # 예: 1 추진배경 -> 1. 추진배경
        s = re.sub(
            r"^(\d{1,2}(?:\.\d{1,2})?)\s+([가-힣A-Za-z].+)$",
            r"\1. \2",
            s
        )

        # 괄호 안 제목형 띄어쓰기
        def fix_spaced_korean_in_parens(m):
            inner = m.group(1)
            if (
                len(inner) <= 50
                and not re.search(r"\d{3,}", inner)
                and re.fullmatch(r"(?:[가-힣A-Za-z&/]\s){1,40}[가-힣A-Za-z&/]", inner)
            ):
                return f"({inner.replace(' ', '')})"
            return m.group(0)

        s = re.sub(
            r"\(((?:[가-힣A-Za-z&/]\s){1,40}[가-힣A-Za-z&/])\)",
            fix_spaced_korean_in_parens,
            s
        )

        # ----------------------------------------------
        # 4-3) 목차형 줄의 잡음 문자 정리
        # ----------------------------------------------
        is_toc_like = bool(re.match(
            r"""^(
                \[.*\]|
                <.*>|
                [ⅠⅡⅢⅣⅤⅥⅦⅧⅨⅩIVX]+[\.．]?\s*|
                제?\d+장\s+|
                \d+(\.\d+)*[\.\)]\s+|
                [가나다라마바사아자차카타파하][\.\)]\s+|
                (붙임|별지|별표|서식)\s*\d*|
                (공통|기능|인터페이스|테스트|보안|품질|제약사항|프로젝트)
            )""",
            s,
            re.VERBOSE
        ))

        if is_toc_like:
            # 목차 안에서만 잡음 제거
            s = re.sub(r"\s*[ḿ⁻]+\s*", " ", s)
            s = re.sub(r"\s*-\s*-\s*(\d{1,3})\s*$", r" - \1", s)
            s = re.sub(r"\s*-\s*-?\s*\d{1,3}(?:\s*[xX])?\s*$", "", s)
            s = re.sub(r"\s*[·\.…]{3,}\s*\d{1,3}\s*$", "", s)

        new_lines.append(s)

    text = "\n".join(new_lines)

    # --------------------------------------------------
    # 5) 빈 줄 정리
    # --------------------------------------------------
    text = re.sub(r"[ ]{2,}", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()

    return text

### 정제함수 적용

In [62]:
# 1. 공기업/공공기관(public)
mask_public_hwp = (df["기관 섹터"] == "공기업/공공기관(public)") & is_hwp
df.loc[mask_public_hwp, "정제텍스트"] = df.loc[mask_public_hwp, "공통정제텍스트"].apply(clean_text_public)

# 2. 연구기관(research)
mask_research_hwp = (df["기관 섹터"] == "연구기관(research)") & is_hwp
df.loc[mask_research_hwp, "정제텍스트"] = df.loc[mask_research_hwp, "공통정제텍스트"].apply(clean_text_research)

# 3. 대학/교육기관(education)
mask_education_hwp = (df["기관 섹터"] == "대학/교육기관(education)") & is_hwp
df.loc[mask_education_hwp, "정제텍스트"] = df.loc[mask_education_hwp, "공통정제텍스트"].apply(clean_text_education)

# 4. 지자체(local)
mask_local_hwp = (df["기관 섹터"] == "지자체(local)") & is_hwp
df.loc[mask_local_hwp, "정제텍스트"] = df.loc[mask_local_hwp, "공통정제텍스트"].apply(clean_text_local)

# 5. 재단/협회/비영리(nonprofit)
mask_nonprofit_hwp = (df["기관 섹터"] == "재단/협회/비영리(nonprofit)") & is_hwp
df.loc[mask_nonprofit_hwp, "정제텍스트"] = df.loc[mask_nonprofit_hwp, "공통정제텍스트"].apply(clean_text_nonprofit)

# 6. 중앙행정기관(central)
mask_central_hwp = (df["기관 섹터"] == "중앙행정기관(central)") & is_hwp
df.loc[mask_central_hwp, "정제텍스트"] = df.loc[mask_central_hwp, "공통정제텍스트"].apply(clean_text_central)

# 7. 민간기업(company)
mask_company_hwp = (df["기관 섹터"] == "민간기업(company)") & is_hwp
df.loc[mask_company_hwp, "정제텍스트"] = df.loc[mask_company_hwp, "공통정제텍스트"].apply(clean_text_company)

In [63]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   공고 번호      100 non-null    str    
 1   공고 차수      82 non-null     float64
 2   사업명        100 non-null    str    
 3   사업 금액      100 non-null    float64
 4   발주 기관      100 non-null    str    
 5   공개 일자      100 non-null    str    
 6   입찰 참여 시작일  100 non-null    str    
 7   입찰 참여 마감일  93 non-null     str    
 8   사업 요약      100 non-null    str    
 9   파일형식       100 non-null    str    
 10  파일명        100 non-null    str    
 11  텍스트        100 non-null    str    
 12  텍스트길이      7 non-null      float64
 13  기관 섹터      100 non-null    str    
 14  공통정제텍스트    100 non-null    str    
 15  정제텍스트      100 non-null    str    
dtypes: float64(3), str(13)
memory usage: 4.3 MB


### 정제후 저장

In [64]:
# 저장
output_path = "/home/bidcoin/data_cleaning3.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"저장 완료: {output_path}")

저장 완료: /home/bidcoin/data_cleaning3.csv
